# Task 3: Correlation between News Sentiment and Stock Movement

This notebook implements sentiment analysis on financial news headlines and measures correlation with stock returns.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from nltk.sentiment import SentimentIntensityAnalyzer
import nltk

nltk.download('vader_lexicon')

## 1. Load Data

In [ ]:
# Update paths as needed
news = pd.read_csv('../data/raw/news.csv')
stocks = pd.read_csv('../data/raw/AAPL.CSV')

news.head(), stocks.head()

## 2. Date Alignment

In [ ]:
news['date'] = pd.to_datetime(news['date'], utc=True)
stocks['Date'] = pd.to_datetime(stocks['Date'])

# Align to trading day (simple forward fill approach)
news['trade_date'] = news['date'].dt.date

# merge strategy placeholder
stocks['trade_date'] = stocks['Date'].dt.date

## 3. Sentiment Analysis (VADER)

In [ ]:
sia = SentimentIntensityAnalyzer()

news['sentiment'] = news['headline'].astype(str).apply(lambda x: sia.polarity_scores(x)['compound'])
news[['headline','sentiment']].head()

## 4. Aggregate Daily Sentiment

In [ ]:
daily_sentiment = news.groupby(['stock','trade_date'])['sentiment'].mean().reset_index()
daily_sentiment.head()

## 5. Calculate Daily Returns

In [ ]:
stocks = stocks.sort_values('Date')
stocks['return'] = stocks['Close'].pct_change() * 100
stocks['trade_date'] = stocks['Date'].dt.date

## 6. Merge Datasets

In [ ]:
merged = pd.merge(daily_sentiment, stocks, on='trade_date')
merged.head()

## 7. Correlation Analysis

In [ ]:
corr = merged['sentiment'].corr(merged['return'])
corr

## 8. Visualization

In [ ]:
plt.scatter(merged['sentiment'], merged['return'])
plt.title(f'Sentiment vs Return (corr={corr:.2f})')
plt.xlabel('Sentiment')
plt.ylabel('Return %')
plt.show()